In [4]:
###FIRST BLOCK GETS EVERYONE INTO ONE TABLE
#DOES THIS BY BRINGING IN BOTH DATASETS (sheet1, sheet 2)
import pandas as pd

# Load the two sheets
sheet1 = pd.read_excel('probe_seqs.xlsx')  # or use sheet index 0
sheet2 = pd.read_excel('interface_data.xlsx')  # or use sheet index 1
#print(sheet1.head())
#print(sheet1.shape)
every_probe = sheet1
# Rename columns if needed to ensure consistent naming (adjust if your column names differ)
sheet1.columns = ['probename', 'sequence', 'start', 'end']
sheet2.columns = ['probename', 'fold_change', 'p_val', 'sRNA']

# Left join: keep all rows from sheet2, match from sheet1
merged = sheet2.merge(sheet1[['probename', 'sequence', 'start', 'end']], 
                      on='probename', 
                      how='left')
# Reorder columns
INTERFACE_DATA_TABLE= merged[['probename', 'fold_change', 'p_val', 'sRNA', 'sequence', 'start', 'end']]
INTERFACE_DATA_TABLE.to_excel('INTDATA_W_STATS.xlsx')
print(INTERFACE_DATA_TABLE.head())
print(INTERFACE_DATA_TABLE.shape)

               probename  fold_change     p_val   sRNA          sequence  \
0  3ETSleuZ_sRNA_probe_0     1.024843  0.909966   True  ATTCTTATCTTTCCCA   
1  3ETSleuZ_sRNA_probe_1     7.316041  0.001012   True   TTATTGCTTTGATTT   
2  3ETSleuZ_sRNA_probe_2     2.289217  0.058387   True    CACGACACTGCTTA   
3  accA_ldcC_UTR_probe_0     3.695827  0.029365  False      AATTACGCGTAA   
4  accA_ldcC_UTR_probe_1     7.962195  0.038762  False    GAACTTTTGCGAAT   

   start   end  
0    0.0  15.0  
1   17.0  31.0  
2   29.0  42.0  
3    0.0  11.0  
4    9.0  22.0  
(6157, 7)


In [ ]:
import pandas as pd 
### ADD CLASH OVERLAPH Hfq CLASH uncovers sRNA-target interaction networks linked to nutrient availability adaptation

###FIRST THINGS FIRST GET THE CLASH DATA LOADED
dfclash = pd.read_excel('hfq_CLASH.xlsx') 
dfclash.columns = ['start','end','strand','pair']

import pandas as pd
import numpy as np

def check_overlap(df1, df2):
    """
    Check if each row in df1 (with 'left'/'right' columns) overlaps with
    any region in df2 (with 'start'/'end' columns).
    
    Returns df1 with a new boolean column 'has_overlap'.
    """
    def row_has_overlap(row):
        # Two intervals [a,b] and [c,d] overlap if a <= d and c <= b
        return ((df2['start'] <= row['right']) & (df2['end'] >= row['left'])).any()
    
    df1['Hfq_CLASH_Overlap'] = df1.apply(row_has_overlap, axis=1)
    return df1
check_overlap(IDT_w_RNE,dfclash)
print(IDT_w_RNE.head())
print(IDT_w_RNE.shape)
#IDT_w_RNE.to_excel('hfq_clash_addition_test.xlsx')

In [ ]:
### MAKE HISTOGRAM OF FOLD CHANGES
import seaborn as sns
fig, ax = plt.subplots(figsize=(8, 6))
df['log2_fc'] = np.log2(df['fold_change'])
df['target'] = df['probename'].str.extract(r'^(.+?)_probe_\d+$')
tgtlist = ['glgC_5_UTR', 'pgaA_5_UTR','nhaA_nhaR_UTR','cstA_5_UTR','miaA_hfq_UTR','flhD_5_UTR','sdia_5_UTR','rpoE_5_UTR','csrA_5_UTR','iraD_5_UTR','mcaS_sRNA','spf_sRNA','gadY_sRNA','gcvB_sRNA']
dftarget = df[df['target'].isin(tgtlist)]
#print(dftarget.shape)
#print(df.head())
df_significant = df['p_val'] < 0.05
df_significant = df[df_significant]
meanFC = np.mean(df_significant['fold_change'])
dftargetsig = dftarget['p_val'] < 0.05
dftargetsig = dftarget[dftargetsig]
meanFCtarget = np.mean(dftargetsig['fold_change'])
print(dftargetsig.shape)
print(dftarget.shape)
print(df.shape)
print(df_significant.shape)
#ax.hist(df['log2_fc'], bins=100, color='steelblue', edgecolor='white', linewidth=0.5)
sns.histplot(df['log2_fc'], ax=ax, color = 'blue',alpha = 0.8)
sns.histplot(df_significant['log2_fc'],ax=ax,color = 'red',alpha = 0.8)
sns.histplot(dftarget['log2_fc'], ax=ax, color = 'green',alpha = 0.7)
ax.legend(['all_probes','p < 0.05','Well Characterized CsrA targets'])
ax.set_xlabel('log2(Fold Change)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Distribution of log2(Fold Change)', fontsize=15)
plt.savefig('log2FC distribution with posctrls.png',dpi = 600)
plt.tight_layout()
plt.show()